# Automatické tréningy pre rôzne n_input a horizonty (DST+1 … DST+6)

Tento notebook načíta dáta **iba raz** a potom postupne vytvorí a natrénuje samostatný model pre každú kombináciu:
- `n_input` ∈ {6, 12, 18, 24, 30, 36, 42, 48}
- `y_col` = `DST+1` … `DST+6`

Model/weighty sa ukladajú do `models/` a súhrn metrík do `results/summary.csv`.


In [1]:
import os
import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from keras.models import Model
from keras.layers import Dense, Input, LSTM, Flatten, TimeDistributed, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Reprodukovateľnosť (voliteľné)
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)


2026-04-10 14:13:51.215703: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os; 
print(os.getcwd())

/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/3_modelovanie/train automate


In [3]:
# =========================
# Nastavenia
# =========================

# Cesty k datasetom (ponechané ako v pôvodnom notebooku)
BASE_PATH = "/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/0_datasety/"
file_path_train = BASE_PATH + "train_omni.csv"
file_path_test  = BASE_PATH + "test_omni.csv"


# Tréningové nastavenia
BATCH_SIZE = 256
EPOCHS = 200
PATIENCE = 25  # early stopping na val_mae

# Kombinácie na tréning
#N_INPUT_LIST = [36, 42, 48]
N_INPUT_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
HORIZONS = [1]

# Výstupné priečinky
MODELS_DIR = "models"
RESULTS_DIR = "results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [4]:
# =========================
# Načítanie dát (iba raz)
# =========================
train_raw = pd.read_csv(file_path_train)
test_raw  = pd.read_csv(file_path_test)

# Skontroluj, aké DST+ stĺpce existujú (pomôže odhaliť preklepy v názvoch)
dst_cols = [c for c in train_raw.columns if c.startswith("DST+")]
print("DST+ stĺpce v train:", dst_cols)
print("DST+ stĺpce v test :", [c for c in test_raw.columns if c.startswith("DST+")])

# čas
if "time1" in train_raw.columns:
    train_raw["time1"] = pd.to_datetime(train_raw["time1"])
if "time1" in test_raw.columns:
    test_raw["time1"] = pd.to_datetime(test_raw["time1"])


DST+ stĺpce v train: ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']
DST+ stĺpce v test : ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']


In [5]:
train_raw

,Unnamed: 0.1,Unnamed: 0,time1,bz_gsm,v,DST,DST+1,DST+2,DST+3,DST+4,DST+5,DST+6
0,0,0,1963-01-01 00:30:00+00:00,-0.2,285.0,-6,-6,-5.0,-5.0,-3.0,-3.0,-6.0
1,1,1,1963-01-01 01:30:00+00:00,-0.2,285.0,-5,-5,-5.0,-3.0,-3.0,-6.0,-8.0
2,2,2,1963-01-01 02:30:00+00:00,-0.2,285.0,-5,-5,-3.0,-3.0,-6.0,-8.0,-9.0
3,3,3,1963-01-01 03:30:00+00:00,-0.2,285.0,-3,-3,-3.0,-6.0,-8.0,-9.0,-6.0
4,4,4,1963-01-01 04:30:00+00:00,-0.2,285.0,-3,-3,-6.0,-8.0,-9.0,-6.0,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
286690,286690,286690,1995-09-15 10:30:00+00:00,-5.6,429.0,-52,-52,-48.0,-40.0,-39.0,-45.0,-48.0
286691,286691,286691,1995-09-15 11:30:00+00:00,-2.9,435.0,-48,-48,-40.0,-39.0,-45.0,-48.0,-45.0
286692,286692,286692,1995-09-15 12:30:00+00:00,-7.8,440.0,-40,-40,-39.0,-45.0,-48.0,-45.0,-41.0
286693,286693,286693,1995-09-15 13:30:00+00:00,-5.6,440.0,-39,-39,-45.0,-48.0,-45.0,-41.0,-41.0


In [6]:
def build_model(n_input: int, n_features: int) -> keras.Model:
    """LSTM model pre multivariačný vstup."""
    inputs = Input(shape=(n_input, n_features))

    x = Bidirectional(
        LSTM(128, return_sequences=True, dropout=0.1, recurrent_dropout=0.1)
    )(inputs)
    x = LSTM(128, return_sequences=True)(x)
    x = TimeDistributed(Dense(1, activation="linear"))(x)
    x = Flatten()(x)
    outputs = Dense(1, activation="linear")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss="mse", optimizer="adam", metrics=["mae"])
    return model


def make_splits(train_df: pd.DataFrame, test_df: pd.DataFrame, y_col: str, predictors=None):
    """Pripraví train/valid/test dáta pre daný y_col (bez NaN)."""

    if predictors is None:
        predictors = ["DST", "v"]   

    features = predictors + [y_col]

    train = train_df[features].copy()
    test = test_df[features].copy()

    # odstránenie NaN
    train = train.dropna().reset_index(drop=True)
    test = test.dropna().reset_index(drop=True)

    # časový split train/valid
    valid_size = int(len(train) * 0.2)

    if valid_size == 0:
        raise ValueError(f"Príliš málo dát po dropna pre {y_col}")

    valid = train.iloc[-valid_size:, :].copy()
    train = train.iloc[:-valid_size, :].copy()

    # X = 2 features
    X_train = train[predictors].values
    y_train = train[y_col].values

    X_val = valid[predictors].values
    y_val = valid[y_col].values

    X_test = test[predictors].values
    y_test = test[y_col].values

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors

In [7]:
def train_one(y_col: str, n_input: int):
    predictors = ["DST", "v"]   

    (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors = make_splits(
        train_raw, test_raw, y_col, predictors=predictors
    )

    train_gen = TimeseriesGenerator(X_train, y_train, length=n_input, batch_size=BATCH_SIZE)
    val_gen   = TimeseriesGenerator(X_val, y_val, length=n_input, batch_size=BATCH_SIZE)
    test_gen  = TimeseriesGenerator(X_test, y_test, length=n_input, batch_size=BATCH_SIZE)

    if len(train_gen) == 0 or len(val_gen) == 0:
        print(f"[SKIP] {y_col}, n_input={n_input} – prázdny generátor")
        return None

    n_features = len(predictors)
    model = build_model(n_input, n_features)

    print("Predictors:", predictors)
    print("X_train shape:", X_train.shape)
    print("First batch X shape:", train_gen[0][0].shape)
    print("Model input shape:", model.input_shape)

    model_path = os.path.join(MODELS_DIR, f"{y_col}_{n_input}H_V.keras")
    checkpoint = ModelCheckpoint(model_path, monitor="val_mae", verbose=1, save_best_only=True, mode="min")
    early = EarlyStopping(monitor="val_mae", mode="min", patience=PATIENCE, restore_best_weights=True)
    callbacks = [checkpoint, early]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        verbose=1,
        callbacks=callbacks
    )

    test_loss, test_mae = model.evaluate(test_gen, verbose=0)

    hist_df = pd.DataFrame(history.history)
    hist_csv = os.path.join(RESULTS_DIR, f"history_{y_col}_{n_input}H_V.csv")
    hist_df.to_csv(hist_csv, index=False)

    return {
        "y_col": y_col,
        "horizon_hours": int(y_col.split("+")[1]),
        "n_input": n_input,
        "best_val_mae": float(np.min(hist_df["val_mae"])) if "val_mae" in hist_df else np.nan,
        "best_val_loss": float(np.min(hist_df["val_loss"])) if "val_loss" in hist_df else np.nan,
        "test_mae": float(test_mae),
        "test_loss": float(test_loss),
        "model_path": model_path,
        "history_path": hist_csv,
        "epochs_ran": int(len(hist_df)),
    }

In [8]:
# =========================
# Spustenie všetkých tréningov
# =========================
summary_rows = []

for h in HORIZONS:
    y_col = f"DST+{h}"

    # ochrana, ak by stĺpec neexistoval
    if y_col not in train_raw.columns or y_col not in test_raw.columns:
        print(f"[SKIP] {y_col} neexistuje v datasete.")
        continue

    for n_input in N_INPUT_LIST:
        print("\n" + "="*80)
        print(f"Trénujem: y_col={y_col}, n_input={n_input}")
        print("="*80)

        row = train_one(y_col, n_input)
        if row is not None:
            summary_rows.append(row)


summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(RESULTS_DIR, "summary1_V.csv")
summary.to_csv(summary_path, index=False)

summary.sort_values(["horizon_hours", "n_input"]).head(20), summary_path



Trénujem: y_col=DST+1, n_input=6
Predictors: ['DST', 'v']
X_train shape: (229356, 2)
First batch X shape: (256, 6, 2)
Model input shape: (None, 6, 2)
Epoch 1/200


/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 284.6702 - mae: 9.3789
Epoch 1: val_mae improved from inf to 5.82390, saving model to models/DST+1_6H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 69s 71ms/step - loss: 284.5275 - mae: 9.3757 - val_loss: 157.0167 - val_mae: 5.8239
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 67.9322 - mae: 4.5087
Epoch 2: val_mae improved from 5.82390 to 4.41528, saving model to models/DST+1_6H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 79s 88ms/step - loss: 67.9136 - mae: 4.5082 - val_loss: 86.5463 - val_mae: 4.4153
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 38.6143 - mae: 3.7834
Epoch 3: val_mae did not improve from 4.41528
896/896 ━━━━━━━━━━━━━━━━━━━━ 75s 84ms/step - loss: 38.6149 - mae: 3.7834 - val_loss: 80.0488 - val_mae: 4.7246
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 38.7444 - mae: 3.8320
Epoch 4: val_mae improved from 4.41528 to 4.02567, saving model to models/DST+1_6H_V.keras
896/896 ━━━━━━━━━━━━━━━━━

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - loss: 237.6170 - mae: 9.0119
Epoch 1: val_mae improved from inf to 5.23137, saving model to models/DST+1_12H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 142s 148ms/step - loss: 237.5177 - mae: 9.0093 - val_loss: 137.3069 - val_mae: 5.2314
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - loss: 64.9526 - mae: 4.8747
Epoch 2: val_mae did not improve from 5.23137
896/896 ━━━━━━━━━━━━━━━━━━━━ 127s 141ms/step - loss: 64.9500 - mae: 4.8746 - val_loss: 97.6819 - val_mae: 5.3488
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - loss: 58.0568 - mae: 4.7489
Epoch 3: val_mae improved from 5.23137 to 5.18081, saving model to models/DST+1_12H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 127s 142ms/step - loss: 58.0486 - mae: 4.7485 - val_loss: 105.0889 - val_mae: 5.1808
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - loss: 68.5655 - mae: 4.8640
Epoch 4: val_mae improved from 5.18081 to 4.63718, saving model to models/DST+1_12H_V.keras
896/896 ━━━

...

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 507ms/step - loss: 355.3628 - mae: 12.2249
Epoch 1: val_mae improved from inf to 6.64693, saving model to models/DST+1_42H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 511s 555ms/step - loss: 355.2366 - mae: 12.2219 - val_loss: 178.6241 - val_mae: 6.6469
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - loss: 127.5088 - mae: 6.7050
Epoch 2: val_mae improved from 6.64693 to 5.40950, saving model to models/DST+1_42H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 483s 539ms/step - loss: 127.4861 - mae: 6.7046 - val_loss: 123.2981 - val_mae: 5.4095
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - loss: 89.9697 - mae: 5.8384
Epoch 3: val_mae improved from 5.40950 to 5.25294, saving model to models/DST+1_42H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 484s 541ms/step - loss: 89.9620 - mae: 5.8382 - val_loss: 88.6131 - val_mae: 5.2529
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - loss: 86.1271 - mae: 5.7865
Epoch 4: val_mae did not improve from 5.25294
896/896

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - loss: 283.5299 - mae: 11.0139
Epoch 1: val_mae improved from inf to 5.61484, saving model to models/DST+1_48H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 341s 367ms/step - loss: 283.4264 - mae: 11.0111 - val_loss: 134.8506 - val_mae: 5.6148
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step - loss: 104.2074 - mae: 6.1086
Epoch 2: val_mae improved from 5.61484 to 5.16864, saving model to models/DST+1_48H_V.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 324s 362ms/step - loss: 104.1836 - mae: 6.1079 - val_loss: 93.8184 - val_mae: 5.1686
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - loss: 59.9251 - mae: 4.7317
Epoch 3: val_mae did not improve from 5.16864
896/896 ━━━━━━━━━━━━━━━━━━━━ 325s 363ms/step - loss: 59.9389 - mae: 4.7321 - val_loss: 171.1390 - val_mae: 8.5035
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - loss: 81.3937 - mae: 5.6276
Epoch 4: val_mae improved from 5.16864 to 4.85799, saving model to models/DST+1_48H_V.keras
896/896

(   y_col  horizon_hours  n_input  best_val_mae  best_val_loss  test_mae  \
 0  DST+1              1        6      3.736403      37.864460  2.592228   
 1  DST+1              1       12      3.675086      36.378567  2.687336   
 2  DST+1              1       18      3.691281      39.111404  2.707864   
 3  DST+1              1       24      3.517004      36.340092  2.439799   
 4  DST+1              1       30      3.861652      39.490906  2.627005   
 5  DST+1              1       36      3.578403      36.947418  2.511167   
 6  DST+1              1       42      3.751244      41.419689  2.563439   
 7  DST+1              1       48      3.589710      34.941288  2.646760   
 
    test_loss                model_path                     history_path  \
 0  16.309931   models/DST+1_6H_V.keras   results/history_DST+1_6H_V.csv   
 1  16.976128  models/DST+1_12H_V.keras  results/history_DST+1_12H_V.csv   
 2  17.770473  models/DST+1_18H_V.keras  results/history_DST+1_18H_V.csv   
 3  15.309

## Poznámky
- Ak chceš presne poradie ako si písala (najprv `DST+1` pre všetky `n_input`, potom `DST+2`, …), tak to presne robí horný loop (horizonty vonkajší, `n_input` vnútorný).
- Ak chceš opačne (pre dané `n_input` spraviť `DST+1..6`), stačí prehodiť poradie cyklov.
